# Part 4 — RL Reward-System Analysis (GSM8K)

This notebook follows the same analysis style as Part 3, but compares three runs:
- Baseline reward (`rl-gsm8k-teammate`)
- Numeric distance reward (`rl-gsm8k-part4-teammate-j-numeric_distance`)
- Calc-consistency reward (`rl-gsm8k-part4-teammate-k-calc_consistency`)

In [ ]:
import os, json, glob, re, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

## 1) Configuration

In [ ]:
# Run this first if needed (from repo root):
# uv run modal run nanochat_modal.py::download_eval_logs_part4

LOG_ROOT = Path('../eval_logs')  # notebook is under dev/
RUNS = {
    'baseline': 'rl-gsm8k-teammate',
    'numeric_distance': 'rl-gsm8k-part4-teammate-j-numeric_distance',
    'calc_consistency': 'rl-gsm8k-part4-teammate-k-calc_consistency',
}

for label, run_name in RUNS.items():
    run_dir = LOG_ROOT / run_name
    files = sorted(run_dir.glob('eval_step_*.json'))
    print(f'{label:>16}: {run_name} -> {len(files)} eval files')

## 2) Load eval logs

In [ ]:
def load_run(run_label, run_name):
    files = sorted((LOG_ROOT / run_name).glob('eval_step_*.json'))
    if not files:
        return pd.DataFrame(), pd.DataFrame()

    rows = []
    passk_rows = []
    for path in files:
        with open(path) as f:
            data = json.load(f)
        step = data['step']
        passk_rows.append({'run_label': run_label, 'run_name': run_name, 'step': step, **data.get('pass@k', {})})
        for rec in data.get('records', []):
            for k, out in enumerate(rec.get('outcomes', [])):
                rows.append({
                    'run_label': run_label,
                    'run_name': run_name,
                    'step': step,
                    'idx': rec.get('idx'),
                    'sample_k': k,
                    'question': rec.get('question', ''),
                    'reference': rec.get('reference', ''),
                    'generated_text': out.get('generated_text', ''),
                    'is_correct': bool(out.get('is_correct', False)),
                })
    return pd.DataFrame(rows), pd.DataFrame(passk_rows)

all_rows = []
all_passk = []
for label, run_name in RUNS.items():
    df_run, passk_run = load_run(label, run_name)
    if not df_run.empty:
        all_rows.append(df_run)
    if not passk_run.empty:
        all_passk.append(passk_run)

df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
passk_df = pd.concat(all_passk, ignore_index=True) if all_passk else pd.DataFrame()

print(f'Loaded outcomes: {len(df):,}')
print(f'Loaded pass@k rows: {len(passk_df):,}')
passk_df.head() if not passk_df.empty else None

## 3) Pass@k curves across reward systems

In [ ]:
if passk_df.empty:
    print('No pass@k data found')
else:
    cols = [c for c in ['pass@1', 'pass@4', 'pass@8'] if c in passk_df.columns]
    fig, axes = plt.subplots(1, len(cols), figsize=(6*len(cols), 4.5))
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        for label, grp in passk_df.groupby('run_label'):
            g = grp.sort_values('step')
            ax.plot(g['step'], g[col], marker='o', label=label)
        ax.set_title(f'{col} over training')
        ax.set_xlabel('step')
        ax.set_ylabel(col)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
        ax.legend()
    plt.tight_layout()
    plt.show()

## 4) Final-step summary table

In [ ]:
if passk_df.empty or df.empty:
    print('Missing data for summary')
else:
    rows = []
    for label in sorted(df['run_label'].unique()):
        p = passk_df[passk_df['run_label'] == label].sort_values('step')
        final_step = int(p['step'].max())
        p_final = p[p['step'] == final_step].iloc[0].to_dict()
        d_final = df[(df['run_label'] == label) & (df['step'] == final_step)]
        sample_acc = d_final['is_correct'].mean() if len(d_final) else np.nan
        row = {
            'run_label': label,
            'final_step': final_step,
            'sample_accuracy_final': sample_acc,
        }
        for key in ['pass@1', 'pass@4', 'pass@8']:
            if key in p_final:
                row[key] = p_final[key]
        rows.append(row)
    summary_df = pd.DataFrame(rows).sort_values('run_label')
    display(summary_df)
    summary_df.to_csv('part4_summary_table.csv', index=False)
    print('Saved part4_summary_table.csv')

## 5) Problem-type and error-type analysis (final step)

In [ ]:
def classify_problem_type(question: str) -> str:
    q = (question or '').lower()
    if re.search(r'\b(miles?|km|kilom|meter|feet|foot|inch|pound|kg|gram|liter|gallon|hour|minute|second|per\s+\w+)\b', q):
        return 'rate/unit conversion'
    if re.search(r'\b(percent|%|discount|tax|tip|interest|markup)\b', q):
        return 'percentage/ratio'
    if re.search(r'\$|\b(cost|price|earn|spend|buy|sell|profit|dollar|cent|pay|wage|salary)\b', q):
        return 'money/commerce'
    if re.search(r'\b(years? old|age|older|younger|born|birthday)\b', q):
        return 'age/time'
    if re.search(r'\b(area|perimeter|volume|rectangle|square|circle|triangle|length|width|height|radius)\b', q):
        return 'geometry/area'
    if re.search(r'\b(how many|number of|total|count|group|team|class|student|people|person|child|children)\b', q):
        return 'counting/combinatorics'
    if re.search(r'\b(fraction|half|third|quarter|divide|split|share|portion)\b', q):
        return 'fractions/division'
    return 'other arithmetic'

def classify_error_type(generated_text: str, reference: str) -> str:
    gen = (generated_text or '').strip()
    if '####' not in gen:
        return 'missing answer marker'
    m = re.search(r'####\s*([\-0-9\.\,]+)', gen)
    if not m:
        return 'malformed answer'
    pred = m.group(1).replace(',', '')
    m_ref = re.search(r'####\s*([\-0-9\.\,]+)', reference or '')
    ref_val = m_ref.group(1).replace(',', '') if m_ref else None
    if '<<' in gen and '>>' in gen:
        return 'calc reasoning error'
    try:
        if ref_val is not None and abs(float(pred) - float(ref_val)) <= 1:
            return 'off-by-one / rounding'
    except Exception:
        return 'non-numeric answer'
    return 'wrong final answer'

if df.empty:
    print('No outcome data')
else:
    final_steps = passk_df.groupby('run_label')['step'].max().to_dict()
    final_df = pd.concat([
        df[(df['run_label'] == label) & (df['step'] == step)]
        for label, step in final_steps.items()
    ], ignore_index=True)

    final_df['problem_type'] = final_df['question'].apply(classify_problem_type)
    incorrect = final_df[~final_df['is_correct']].copy()
    incorrect['error_type'] = incorrect.apply(
        lambda r: classify_error_type(r['generated_text'], r['reference']), axis=1
    )

    print('Incorrect by run:')
    display(incorrect.groupby('run_label').size().rename('n_incorrect').reset_index())

    err_frac = (
        incorrect.groupby(['run_label', 'error_type']).size()
        .groupby(level=0).apply(lambda s: s / s.sum())
        .reset_index(name='fraction')
    )
    display(err_frac.head(20))

In [ ]:
if 'incorrect' in globals() and len(incorrect):
    pivot = pd.crosstab(incorrect['run_label'], incorrect['error_type'], normalize='index')
    plt.figure(figsize=(10, 4.5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd')
    plt.title('Final-step error type fractions by reward system')
    plt.xlabel('error type')
    plt.ylabel('run')
    plt.tight_layout()
    plt.show()

    ptype_acc = final_df.groupby(['run_label', 'problem_type'])['is_correct'].mean().reset_index()
    plt.figure(figsize=(11, 5))
    sns.barplot(data=ptype_acc, x='problem_type', y='is_correct', hue='run_label')
    plt.xticks(rotation=30, ha='right')
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    plt.title('Accuracy by problem type (final eval step)')
    plt.tight_layout()
    plt.show()

## 6) Qualitative samples

In [ ]:
if 'final_df' not in globals() or final_df.empty:
    print('No final-step data')
else:
    for label in sorted(final_df['run_label'].unique()):
        sub = final_df[final_df['run_label'] == label]
        print(f'\n{'='*80}\nRUN: {label}\n{'='*80}')
        corr = sub[sub['is_correct']].drop_duplicates('idx').head(2)
        inc = sub[~sub['is_correct']].drop_duplicates('idx').head(2)
        for _, row in pd.concat([corr, inc]).iterrows():
            print('\nQ:', textwrap.shorten(str(row['question']), width=140, placeholder='...'))
            print('Correct:', bool(row['is_correct']))
            print('Output:', textwrap.shorten(str(row['generated_text']), width=180, placeholder='...'))

## 7) Writeup notes template

Use this structure in your report:
1. **Curve comparison**: Which reward reached higher pass@1/pass@8 and how fast?
2. **Error profile shifts**: Which reward reduced marker/format errors vs arithmetic/setup errors?
3. **Problem-type shifts**: Any categories where one reward helps/hurts?
4. **Tradeoffs**: Did denser rewards improve sample efficiency but hurt final exactness?
5. **Conclusion**: Best reward system under fixed compute budget, and why.